**Identificações:**
* Enrico Minto Spanier - enrico.spanier@mackenzista.com.br - 10419775
* Guilherme Vecchi Bonotti Freitas Silveira - guilherme.silveir@mackenzista.com.br - 10418517
* Alexandre Luppi Severo e Silva - alexandreluppi.silva@mackenzista.com.br - 10419724

**Síntese do conteúdo:** Este notebook realiza a importação dos dados brutos do dataset Olist, análise exploratória (distribuição de compras e avaliações), tratamento de valores nulos e estruturação da matriz de interações (usuário-produto) para posterior treinamento da rede neural de recomendação.

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuração visual dos gráficos
sns.set_theme(style="whitegrid")

### 1. Carregamento dos Dados
Para este projeto, utilizaremos três tabelas principais do dataset Olist: Pedidos, Itens do Pedido e Avaliações.
*(Nota: Certifique-se de que os arquivos .csv estejam na pasta `dados/` ou ajuste os caminhos abaixo)*

In [ ]:
# Carregamento dos datasets brutos
orders = pd.read_csv('dados/olist_orders_dataset.csv')
items = pd.read_csv('dados/olist_order_items_dataset.csv')
reviews = pd.read_csv('dados/olist_order_reviews_dataset.csv')

# Mesclando os dados para criar a relação: Usuário -> Pedido -> Produto -> Avaliação
df_merged = orders[['order_id', 'customer_id']].merge(items[['order_id', 'product_id']], on='order_id')
df_merged = df_merged.merge(reviews[['order_id', 'review_score']], on='order_id')

print(f"Total de interações registradas: {df_merged.shape[0]}")
df_merged.head()

### 2. Análise Exploratória de Dados (EDA)
Vamos entender como as avaliações estão distribuídas e a densidade de compras.

In [ ]:
# Distribuição das notas de avaliação
plt.figure(figsize=(8, 5))
sns.countplot(data=df_merged, x='review_score', palette='viridis')
plt.title('Distribuição das Notas de Avaliação (1 a 5)')
plt.xlabel('Nota')
plt.ylabel('Quantidade')
plt.show()

# Verificando produtos mais comprados
top_products = df_merged['product_id'].value_counts().head(10)
print("Top 10 Produtos Mais Vendidos (IDs):")
print(top_products)

### 3. Preparação e Limpeza dos Dados
Para redes neurais (Embeddings), os IDs de usuários e produtos devem ser convertidos em índices numéricos contínuos (0, 1, 2...). Vamos remover dados faltantes e criar esses índices.

In [ ]:
# Removendo linhas com valores nulos nas colunas cruciais
df_clean = df_merged.dropna(subset=['customer_id', 'product_id', 'review_score']).copy()

# Criando índices numéricos para as categorias (Label Encoding)
df_clean['user_idx'] = df_clean['customer_id'].astype('category').cat.codes.values
df_clean['item_idx'] = df_clean['product_id'].astype('category').cat.codes.values

# Selecionando colunas finais para o modelo
df_final = df_clean[['user_idx', 'item_idx', 'review_score']]

print(f"Total de Usuários Únicos: {df_final['user_idx'].nunique()}")
print(f"Total de Produtos Únicos: {df_final['item_idx'].nunique()}")

# Exportando o dataset preparado para a N2
df_final.to_csv('dados/dataset_preparado.csv', index=False)
print("Dataset preparado exportado com sucesso!")